<a href="https://colab.research.google.com/github/MehrasAmirhakimi/GPU-MHD-Orszag-Tang-2D/blob/main/notebooks/colab_gpu_port.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUDA port on a Colab GPU

Builds the C++ CPU backend and the CUDA backend, runs the Orszag-Tang vortex
on the GPU, checks it field by field against the NumPy reference state in the
repository, then times all three implementations on the same machine.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

In [1]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!nvcc --version | tail -2

name, compute_cap, memory.total [MiB]
Tesla T4, 7.5, 15360 MiB
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
import os
REPO = "https://github.com/MehrasAmirhakimi/GPU-MHD-Orszag-Tang-2D.git"
if not os.path.isdir("GPU-MHD-Orszag-Tang-2D"):
    !git clone -q {REPO}
%cd GPU-MHD-Orszag-Tang-2D
!git pull -q
!git log --oneline -1

/content/GPU-MHD-Orszag-Tang-2D
51c218c (HEAD -> main, origin/main, origin/HEAD) Stage 2: CUDA port and C++ CPU backend, validated against NumPy


## Build

`ARCH=native` lets nvcc detect the card. If that fails on an older nvcc, name
it instead: `sm_75` for a T4, `sm_80` for an A100, `sm_89` for an L4.

In [3]:
!make -C cuda cpu gpu ARCH=native

make: Entering directory '/content/GPU-MHD-Orszag-Tang-2D/cuda'
g++ -O2 -std=c++17 -Wall -Wextra -fopenmp main.cpp backend_cpu.cpp -o mhd2d_cpu
nvcc -O2 -std=c++17 -arch=native --fmad=true main.cpp backend_cuda.cu -o mhd2d_gpu
make: Leaving directory '/content/GPU-MHD-Orszag-Tang-2D/cuda'


## Validation

Same grid, same diffusivities and the same end time as the stored reference
`ot_state.npz` (256², t = 0.5). `compare.py` exits with PASS if every field
agrees with NumPy to 1e-10 relative to that field's size; a correct port lands
near 1e-13.

If it fails, rebuild with `make -C cuda gpu DEBUG_SYNC=1` and run again: that
synchronises after every kernel, so an error is reported at the kernel that
caused it.

In [4]:
!cd cuda && ./mhd2d_gpu --nx 256 --tmax 0.5 --every 40
!python compare.py ot_state.npz cuda/ot_state_gpu.npy

grid 256 x 256   nu = 0.002  eta = 0.002  chi = 0.002
backend cuda, Tesla T4, sm_75
t = 0.000  Etot = 4.38888889  divB = 3.83e-13
t = 0.149  dt = 3.67e-03  Etot = 4.38888891  rho_min = 2.6934  divB = 3.83e-13
t = 0.294  dt = 3.59e-03  Etot = 4.38888896  rho_min = 2.4853  divB = 3.68e-13
t = 0.436  dt = 3.54e-03  Etot = 4.38888057  rho_min = 2.2280  divB = 4.76e-13
t = 0.500  dt = 9.19e-05  Etot = 4.38885906  rho_min = 2.1103  divB = 3.76e-13

139 steps in 0.264 s  (0.02893 us per point-step, 34.6 Mpoint-steps/s)
energy drift  6.796e-06
mass drift    3.797e-09
max |div B|   3.758e-13
pressure      min 1.0544   max 5.6853
wrote ot_state_gpu.npy, ot_diagnostics_gpu.csv
grid 256 x 256, tolerance 1e-10
 field     max|ref|    max|diff|   normalised
 lnrho   1.7484e+00   2.8866e-14   1.6509e-14
    ux   1.0687e+00   2.8033e-14   2.6232e-14
    uy   1.2984e+00   2.6867e-14   2.0692e-14
    aa   1.4969e+00   1.3323e-15   8.9004e-16
    ss   9.4428e-03   7.3379e-16   7.7709e-14

worst normalised

## Timing

A fixed number of steps per grid size, so every run does the same work per
step. The NumPy figure comes from `run_ot.py` over a short run, and the C++
CPU backend uses every core Colab provides. All three are float64.

This is the naive port: global memory only, no tiling, no kernel fusion. It
is the baseline the optimisation stage has to beat.

In [5]:
import os, re, subprocess

def per_point_us(cmd, cwd="."):
    out = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return float(re.search(r"\(([0-9.eE+-]+) us per point-step", out.stdout).group(1))

rows = []

# NumPy reference, single core. Run in its own folder: run_ot.py writes
# ot_state.npz, and it must not overwrite the reference state in the repo root.
os.makedirs("timing_numpy", exist_ok=True)
us = per_point_us("python ../run_ot.py --nx 256 --tmax 0.04 --no-plot", cwd="timing_numpy")
rows.append(("NumPy", 256, us))

# C++ CPU backend, OpenMP
for n, steps in [(256, 50), (512, 20)]:
    us = per_point_us(f"./mhd2d_cpu --nx {n} --steps {steps} --every 0 --quiet "
                      f"--out /dev/null --csv /dev/null", cwd="cuda")
    rows.append(("C++ CPU", n, us))

# CUDA backend
for n, steps in [(256, 200), (512, 100), (1024, 50), (2048, 20)]:
    us = per_point_us(f"./mhd2d_gpu --nx {n} --steps {steps} --every 0 --quiet "
                      f"--out /dev/null --csv /dev/null", cwd="cuda")
    rows.append(("CUDA", n, us))

numpy_us = rows[0][2]
print("| backend | grid | us per point-step | Mpoint-steps/s | speed-up over NumPy |")
print("| --- | --- | --- | --- | --- |")
for name, n, us in rows:
    print(f"| {name} | {n}^2 | {us:.4g} | {1/us:.1f} | {numpy_us/us:.0f}x |")

| backend | grid | us per point-step | Mpoint-steps/s | speed-up over NumPy |
| --- | --- | --- | --- | --- |
| NumPy | 256^2 | 2.19 | 0.5 | 1x |
| C++ CPU | 256^2 | 1.05 | 1.0 | 2x |
| C++ CPU | 512^2 | 1.348 | 0.7 | 2x |
| CUDA | 256^2 | 0.03346 | 29.9 | 65x |
| CUDA | 512^2 | 0.02064 | 48.4 | 106x |
| CUDA | 1024^2 | 0.0206 | 48.5 | 106x |
| CUDA | 2048^2 | 0.0207 | 48.3 | 106x |


Copy the table above into the README, together with the GPU name from the
first cell.